# Lumpy Skin Disease Classification

I rewrote this study notebook to keep the evaluation honest. The original metadata columns looked useful at first, but their missing-value pattern was almost the same as the target, so I dropped them before modeling.

I kept the project focused on three standard classifiers: KNN, Random Forest, and a linear SVM. My goal here was to compare them on the same split and judge them with positive-class F1 and balanced accuracy instead of plain accuracy alone.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
plt.style.use('ggplot')
pd.set_option('display.max_columns', 50)

In [ ]:
def first_existing_path(*candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(candidates[0])

data_path = first_existing_path(
    '../data/Lumpy_skin_disease_data.csv',
    'data/Lumpy_skin_disease_data.csv',
    'projects/lumpy-skin-disease-classification/data/Lumpy_skin_disease_data.csv',
)
if not data_path.exists():
    raise FileNotFoundError(f'Could not find dataset at {data_path.resolve()}')

df = pd.read_csv(data_path)
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## First checks

I wanted to understand two things before fitting anything: how much missing data we have, and whether any missingness is suspicious enough to leak the label.

In [ ]:
missing_summary = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename('missing_pct')
    .to_frame()
)
display(missing_summary)

class_counts = df['lumpy'].value_counts().sort_index().rename(index={0: 'No LSD', 1: 'LSD reported'})
display(class_counts.to_frame(name='count'))

In [ ]:
leakage_check = pd.DataFrame({
    'missing_pct': df[['region', 'country', 'reportingDate']].isna().mean().mul(100).round(2),
    'missing_when_lumpy_0': [df.loc[df['lumpy'] == 0, col].isna().mean() for col in ['region', 'country', 'reportingDate']],
    'missing_when_lumpy_1': [df.loc[df['lumpy'] == 1, col].isna().mean() for col in ['region', 'country', 'reportingDate']],
}, index=['region', 'country', 'reportingDate'])
leakage_check[['missing_when_lumpy_0', 'missing_when_lumpy_1']] = leakage_check[['missing_when_lumpy_0', 'missing_when_lumpy_1']].mul(100).round(2)
display(leakage_check)

`region`, `country`, and `reportingDate` are all missing for every class-0 row, while class-1 rows usually have values there. That means the model could guess the target from missingness alone, so I removed those columns instead of pretending they were safe features.

In [ ]:
clean_df = df.drop(columns=['region', 'country', 'reportingDate']).copy()
clean_df['dominant_land_cover'] = clean_df['dominant_land_cover'].astype('category')

numeric_preview = clean_df.describe().T[['mean', 'std', 'min', 'max']].round(2)
display(numeric_preview)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['lumpy'].value_counts().sort_index()
axes[0].bar(['No LSD', 'LSD reported'], counts.values, color=['#4c78a8', '#e45756'])
axes[0].set_title('Class balance')
axes[0].set_xlabel('Lumpy skin disease label')
axes[0].set_ylabel('Count')

negative_sample = df[df['lumpy'] == 0].sample(n=3000, random_state=RANDOM_STATE)
positive_rows = df[df['lumpy'] == 1]
# I sampled the large negative class here so the outbreak locations stay readable.
axes[1].scatter(negative_sample['x'], negative_sample['y'], alpha=0.35, s=15, color='#4c78a8', label='No LSD')
axes[1].scatter(positive_rows['x'], positive_rows['y'], alpha=0.55, s=15, color='#e45756', label='LSD reported')
axes[1].set_title('Reported cases by location')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].legend()

plt.tight_layout()
plt.show()

## Modeling plan

I kept the original class imbalance because it reflects the dataset, but I evaluated models with positive-class F1 and balanced accuracy so a high score would actually mean something. Hyperparameter tuning happens only on the training split through stratified cross-validation.

In [ ]:
X = clean_df.drop(columns='lumpy')
y = clean_df['lumpy']

categorical_features = ['dominant_land_cover']
numeric_features = [col for col in X.columns if col not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

scaled_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ]
)

model_spaces = {
    'KNN': {
        'pipeline': Pipeline([
            ('preprocess', scaled_preprocessor),
            ('model', KNeighborsClassifier()),
        ]),
        'params': {
            'model__n_neighbors': [5, 11, 21],
            'model__weights': ['uniform', 'distance'],
            'model__p': [1, 2],
        },
    },
    'Random Forest': {
        'pipeline': Pipeline([
            ('preprocess', tree_preprocessor),
            ('model', RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=1,
                class_weight='balanced_subsample',
            )),
        ]),
        'params': {
            'model__n_estimators': [200],
            'model__max_depth': [None, 12],
            'model__min_samples_leaf': [1, 5],
        },
    },
    'Linear SVM': {
        'pipeline': Pipeline([
            ('preprocess', scaled_preprocessor),
            ('model', LinearSVC(
                random_state=RANDOM_STATE,
                class_weight='balanced',
                dual=False,
                max_iter=5000,
            )),
        ]),
        'params': {
            'model__C': [0.1, 1.0, 3.0],
        },
    },
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f'Train rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')

In [ ]:
results = []
best_models = {}
confusion_matrices = {}
classification_reports = {}

for model_name, config in model_spaces.items():
    search = GridSearchCV(
        estimator=config['pipeline'],
        param_grid=config['params'],
        scoring='f1',
        cv=cv,
        n_jobs=1,
        refit=True,
    )
    search.fit(X_train, y_train)

    preds = search.predict(X_test)
    report = classification_report(y_test, preds, output_dict=True)

    best_models[model_name] = search.best_estimator_
    confusion_matrices[model_name] = confusion_matrix(y_test, preds)
    classification_reports[model_name] = pd.DataFrame(report).T
    results.append({
        'model': model_name,
        'best_params': search.best_params_,
        'cv_f1': search.best_score_,
        'test_precision': precision_score(y_test, preds),
        'test_recall': recall_score(y_test, preds),
        'test_f1': f1_score(y_test, preds),
        'test_balanced_accuracy': balanced_accuracy_score(y_test, preds),
    })

results_df = pd.DataFrame(results).sort_values('test_f1', ascending=False).reset_index(drop=True)
results_df[['cv_f1', 'test_precision', 'test_recall', 'test_f1', 'test_balanced_accuracy']] = (
    results_df[['cv_f1', 'test_precision', 'test_recall', 'test_f1', 'test_balanced_accuracy']].round(4)
)

display(results_df)

In [ ]:
for model_name in results_df['model']:
    print()
    print(model_name)
    print('-' * len(model_name))
    display(classification_reports[model_name].round(4))

Random Forest gave me the strongest balance in this run, landing near `0.89` test F1 and recovering most positive cases. KNN stayed competitive at about `0.86` test F1, while the linear SVM found many positives but paid for that with much lower precision.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, model_name in zip(axes, results_df['model']):
    cm = confusion_matrices[model_name]
    image = ax.imshow(cm, cmap='Blues')
    ax.set_title(model_name)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_xticks([0, 1], ['0', '1'])
    ax.set_yticks([0, 1], ['0', '1'])
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha='center', va='center', color='black')

fig.colorbar(image, ax=axes, shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
rf_model = best_models['Random Forest']
rf_feature_names = rf_model.named_steps['preprocess'].get_feature_names_out()
rf_importances = pd.Series(
    rf_model.named_steps['model'].feature_importances_,
    index=rf_feature_names,
).sort_values(ascending=False)

top_features = rf_importances.head(12).sort_values()
plt.figure(figsize=(8, 6))
plt.barh(top_features.index, top_features.values, color='#4c78a8')
plt.title('Random Forest feature importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

display(rf_importances.head(12).round(4).to_frame(name='importance'))

## What I learned

The biggest lesson here was not about tuning a model. It was noticing that three metadata columns were almost direct shortcuts to the label because of how missing values were recorded.

After removing that leakage, the climate and geography features still carried useful signal. In my run, Random Forest handled that mix best, and the strongest numeric features were precipitation, longitude, and the temperature variables.